# International Student Enrollment and S&A Funding Research

## Purpose

Investigate how international student enrollment relates to Seattle Central's
overall enrollment and Services & Activities (S&A) funding.

This analysis distinguishes between:

- international students generally;
- International Contract students;
- state-funded international students; and
- resident/nonresident state-funded students.

## Research Questions

1. How many international students attended Seattle Central in each dashboard year?
2. Are international students included in the annual all-source headcount?
3. How many international students are coded as International Contract?
4. How many are recorded as state-funded?
5. How does tuition/revenue treatment differ between these groups?
6. Where is International Contract revenue recorded?
7. Does any International Contract revenue ultimately transfer to Fund 522?
8. How should international students be represented when interpreting S&A funding?

## Dashboard Years

- 2023–24
- 2024–25
- 2025–26

## Important Distinction

"International student" and "International Contract student" are not
necessarily equivalent categories.

The analysis must not assume that all international students use the
International Contract funding classification.

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

INTERNATIONAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "manual"
    / "international_enrollment_context.csv"
)

international = pd.read_csv(
    INTERNATIONAL_PATH
)

print("INTERNATIONAL ENROLLMENT CONTEXT")
print("=" * 50)

print(f"Rows: {len(international):,}")
print(f"Columns: {len(international.columns):,}")

display(international)

INTERNATIONAL ENROLLMENT CONTEXT
Rows: 3
Columns: 9


,academic_year,total_headcount,international_all_funds,international_state_fund,international_contract_fund,international_contract_program_all,international_contract_program_contract,status,notes
0,2023-24,12088,1058,32,907,911,911,final,Annual headcount; Seattle Central/SVI
1,2024-25,12378,1132,26,990,994,994,final,Annual headcount; Seattle Central/SVI
2,2025-26,11672,1035,11,924,927,927,provisional,Most recent academic year; SBCTC notes final d...


In [3]:
# =========================================================
# INTERNATIONAL ENROLLMENT VALIDATION
# =========================================================

print("INTERNATIONAL ENROLLMENT VALIDATION")
print("=" * 50)


# 1. EXPECTED ACADEMIC YEARS

expected_years = {
    "2023-24",
    "2024-25",
    "2025-26",
}

actual_years = set(international["academic_year"])

years_correct = actual_years == expected_years

print(f"Academic years correct: {years_correct}")


# 2. DUPLICATE RECORDS

duplicate_years = international.duplicated(
    subset=["academic_year"]
).sum()

print(f"Duplicate years: {duplicate_years}")


# 3. MISSING REQUIRED VALUES

required_columns = [
    "academic_year",
    "total_headcount",
    "international_all_funds",
    "international_state_fund",
    "international_contract_fund",
    "international_contract_program_all",
    "international_contract_program_contract",
    "status",
]

missing_values = international[
    required_columns
].isna().sum()

print("\nMissing required values:")
print(missing_values)


# 4. VALIDATE STUDENT COUNTS

count_columns = [
    "total_headcount",
    "international_all_funds",
    "international_state_fund",
    "international_contract_fund",
    "international_contract_program_all",
    "international_contract_program_contract",
]

counts = international[count_columns].apply(
    pd.to_numeric, errors="coerce"
)

counts_valid = (
    counts.notna().all().all()
    and counts.ge(0).all().all()
    and counts.mod(1).eq(0).all().all()
)

population_check = (
    international["international_all_funds"]
    <= international["total_headcount"]
).all()

fund_filter_check = (
    counts["international_state_fund"]
    .le(counts["international_all_funds"]).all()
    and
    counts["international_contract_fund"]
    .le(counts["international_all_funds"]).all()
)

print(f"\nStudent counts valid: {counts_valid}")
print(f"International <= total: {population_check}")
print(f"Funding filters within total: {fund_filter_check}")


# 5. CONTRACT PROGRAM CONSISTENCY

contract_match = (
    international["international_contract_program_all"]
    ==
    international["international_contract_program_contract"]
).all()

print(f"Contract program counts match: {contract_match}")


# 6. SOURCE REPORTING DIFFERENCE
# Record this discrepancy; do not force equality.

international["contract_group_difference"] = (
    international["international_contract_program_all"]
    -
    international["international_contract_fund"]
)

print("\nContract reporting differences:")

display(
    international[
        [
            "academic_year",
            "contract_group_difference",
        ]
    ]
)


# 7. DATA STATUS

expected_status = {
    "2023-24": "final",
    "2024-25": "final",
    "2025-26": "provisional",
}

status_check = all(
    international.loc[
        international["academic_year"] == year,
        "status",
    ].iloc[0] == status
    for year, status in expected_status.items()
)

print(f"\nReporting statuses correct: {status_check}")


# 8. CROSS-CHECK MAIN ENROLLMENT DATA

enrollment_reference = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "manual"
    / "enrollment_context.csv"
)

crosscheck = international[
    ["academic_year", "total_headcount"]
].merge(
    enrollment_reference[
        ["academic_year", "total_students"]
    ],
    on="academic_year",
    how="left",
    validate="one_to_one",
)

enrollment_match = (
    crosscheck["total_headcount"]
    ==
    crosscheck["total_students"]
).all()

print(f"Matches main enrollment data: {enrollment_match}")


# FINAL VALIDATION

validation_passed = all([
    years_correct,
    duplicate_years == 0,
    missing_values.sum() == 0,
    counts_valid,
    population_check,
    fund_filter_check,
    contract_match,
    status_check,
    enrollment_match,
])

print("\n" + "=" * 50)

print(
    "VALIDATION PASSED"
    if validation_passed
    else "VALIDATION FAILED"
)

INTERNATIONAL ENROLLMENT VALIDATION
Academic years correct: True
Duplicate years: 0

Missing required values:
academic_year                              0
total_headcount                            0
international_all_funds                    0
international_state_fund                   0
international_contract_fund                0
international_contract_program_all         0
international_contract_program_contract    0
status                                     0
dtype: int64

Student counts valid: True
International <= total: True
Funding filters within total: True
Contract program counts match: True

Contract reporting differences:


,academic_year,contract_group_difference
0,2023-24,4
1,2024-25,4
2,2025-26,3



Reporting statuses correct: True
Matches main enrollment data: True

VALIDATION PASSED


In [4]:
# =========================================================
# INTERNATIONAL ENROLLMENT ANALYSIS
# =========================================================

international_summary = (
    international
    .sort_values("academic_year")
    .copy()
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 1. INTERNATIONAL SHARE OF TOTAL ENROLLMENT
# ---------------------------------------------------------

international_summary["international_share_pct"] = (
    international_summary["international_all_funds"]
    / international_summary["total_headcount"]
    * 100
)


# ---------------------------------------------------------
# 2. FUNDING CATEGORY SHARES
# ---------------------------------------------------------

# These describe filtered populations, not exclusive groups.
# A student may appear under multiple funding categories.

international_summary["intl_state_share_pct"] = (
    international_summary["international_state_fund"]
    / international_summary["international_all_funds"]
    * 100
)

international_summary["intl_contract_share_pct"] = (
    international_summary["international_contract_fund"]
    / international_summary["international_all_funds"]
    * 100
)


# ---------------------------------------------------------
# 3. INTERNATIONAL CONTRACT SHARE OF TOTAL
# ---------------------------------------------------------

international_summary["contract_program_share_pct"] = (
    international_summary["international_contract_program_all"]
    / international_summary["total_headcount"]
    * 100
)


# ---------------------------------------------------------
# 4. YEAR-OVER-YEAR ENROLLMENT CHANGES
# ---------------------------------------------------------

international_summary["intl_yoy_change"] = (
    international_summary["international_all_funds"]
    .diff()
)

international_summary["intl_yoy_pct"] = (
    international_summary["international_all_funds"]
    .pct_change(fill_method=None)
    * 100
)

international_summary["contract_yoy_change"] = (
    international_summary["international_contract_program_all"]
    .diff()
)

international_summary["contract_yoy_pct"] = (
    international_summary["international_contract_program_all"]
    .pct_change(fill_method=None)
    * 100
)


# ---------------------------------------------------------
# 5. DISPLAY RESULTS
# ---------------------------------------------------------

percentage_columns = [
    "international_share_pct",
    "intl_state_share_pct",
    "intl_contract_share_pct",
    "contract_program_share_pct",
    "intl_yoy_pct",
    "contract_yoy_pct",
]

international_summary[percentage_columns] = (
    international_summary[percentage_columns]
    .round(2)
)


print("INTERNATIONAL ENROLLMENT SHARES")
print("=" * 50)

display(
    international_summary[
        [
            "academic_year",
            "international_all_funds",
            "international_share_pct",
            "intl_state_share_pct",
            "intl_contract_share_pct",
            "contract_program_share_pct",
        ]
    ]
)


print("\nYEAR-OVER-YEAR ENROLLMENT CHANGES")
print("=" * 50)

display(
    international_summary[
        [
            "academic_year",
            "international_all_funds",
            "intl_yoy_change",
            "intl_yoy_pct",
            "international_contract_program_all",
            "contract_yoy_change",
            "contract_yoy_pct",
            "status",
        ]
    ]
)

INTERNATIONAL ENROLLMENT SHARES


,academic_year,international_all_funds,international_share_pct,intl_state_share_pct,intl_contract_share_pct,contract_program_share_pct
0,2023-24,1058,8.75,3.02,85.73,7.54
1,2024-25,1132,9.15,2.30,87.46,8.03
2,2025-26,1035,8.87,1.06,89.28,7.94



YEAR-OVER-YEAR ENROLLMENT CHANGES


,academic_year,international_all_funds,intl_yoy_change,intl_yoy_pct,international_contract_program_all,contract_yoy_change,contract_yoy_pct,status
0,2023-24,1058,NaN,NaN,911,NaN,NaN,final
1,2024-25,1132,74.0,6.99,994,83.0,9.11,final
2,2025-26,1035,-97.0,-8.57,927,-67.0,-6.74,provisional


In [5]:
# =========================================================
# FULL-PERIOD INTERNATIONAL ENROLLMENT CHANGE
# =========================================================

start = international_summary.loc[
    international_summary["academic_year"] == "2023-24"
].iloc[0]

end = international_summary.loc[
    international_summary["academic_year"] == "2025-26"
].iloc[0]


# 1. OVERALL INTERNATIONAL ENROLLMENT

intl_change = (
    end["international_all_funds"]
    - start["international_all_funds"]
)

intl_change_pct = (
    intl_change
    / start["international_all_funds"]
    * 100
)


# 2. INTERNATIONAL CONTRACT ENROLLMENT

contract_change = (
    end["international_contract_program_all"]
    - start["international_contract_program_all"]
)

contract_change_pct = (
    contract_change
    / start["international_contract_program_all"]
    * 100
)


# 3. CHANGE IN SHARE OF TOTAL ENROLLMENT

share_change_pp = (
    end["international_share_pct"]
    - start["international_share_pct"]
)


# 4. DISPLAY RESULTS

print("FULL-PERIOD ENROLLMENT CHANGE")
print("=" * 50)

print("International students:")
print(f"Net change: {intl_change:+,.0f}")
print(f"Percent change: {intl_change_pct:+.2f}%")

print("\nInternational Contract students:")
print(f"Net change: {contract_change:+,.0f}")
print(f"Percent change: {contract_change_pct:+.2f}%")

print("\nInternational share of total enrollment:")
print(f"Change: {share_change_pp:+.2f} percentage points")

FULL-PERIOD ENROLLMENT CHANGE
International students:
Net change: -23
Percent change: -2.17%

International Contract students:
Net change: +16
Percent change: +1.76%

International share of total enrollment:
Change: +0.12 percentage points


In [7]:
# =========================================================
# FINAL VALIDATION AND EXPORT
# =========================================================

print("FINAL INTERNATIONAL ENROLLMENT CHECK")
print("=" * 50)

# 1. Check expected years and data completeness

expected_years = {
    "2023-24",
    "2024-25",
    "2025-26",
}

required_columns = [
    "academic_year",
    "total_headcount",
    "international_all_funds",
    "international_state_fund",
    "international_contract_fund",
    "international_contract_program_all",
    "international_contract_program_contract",
    "international_share_pct",
    "intl_state_share_pct",
    "intl_contract_share_pct",
    "contract_program_share_pct",
    "status",
]

years_valid = (
    set(international_summary["academic_year"])
    == expected_years
)

no_duplicates = (
    not international_summary["academic_year"]
    .duplicated().any()
)

no_missing = (
    international_summary[required_columns]
    .notna().all().all()
)

differences_valid = (
    international_summary["contract_group_difference"]
    .tolist() == [4, 4, 3]
)

final_check = all([
    validation_passed,
    years_valid,
    no_duplicates,
    no_missing,
    differences_valid,
])

print(f"Prior validation passed: {validation_passed}")
print(f"Years valid: {years_valid}")
print(f"No duplicates: {no_duplicates}")
print(f"No missing required data: {no_missing}")
print(f"Reporting differences preserved: {differences_valid}")


# =========================================================
# EXPORT PROCESSED DATA
# =========================================================

if final_check:

    output = international_summary.copy()

    # Convert year-over-year changes to nullable integers
    for col in ["intl_yoy_change", "contract_yoy_change"]:
        output[col] = output[col].astype("Int64")

    OUTPUT_PATH = (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "international_enrollment_summary.csv"
    )

    output.to_csv(
        OUTPUT_PATH,
        index=False
    )

    print("\nALL CHECKS PASSED")
    print(f"Saved: {OUTPUT_PATH}")
    print(f"Rows exported: {len(output)}")

else:
    print("\nVALIDATION FAILED — export skipped")
    
    


FINAL INTERNATIONAL ENROLLMENT CHECK
Prior validation passed: True
Years valid: True
No duplicates: True
No missing required data: True
Reporting differences preserved: True

ALL CHECKS PASSED
Saved: /Users/mainguyen/GitHub/seattle-central-sa-budget-dashboard/data/processed/international_enrollment_summary.csv
Rows exported: 3



## International Contract Revenue and S&A Accounting

### Research Objective

Determine whether International Contract payments contribute to Seattle Central's S&A budget, directly or through internal transfers.

### Verified Statewide Accounting Rules

**Standard S&A tuition**
- 96.5%: Fund 522, Class 264 (S&A)
- 3.5%: Fund 860, Class 279 (institutional financial aid)

**International Contract**
- Fee-in-lieu of tuition
- Initial accounting: Fund 146
- Revenue account: 4021065
- SBCTC remittance: 2.5% beginning FY2024

### Seattle Colleges-Specific Findings

Seattle Colleges identifies Fund 146 (grants/contracts) and Fund 522 (student activity fees) separately.

Historical Seattle Central student-government records show that collecting additional S&A fees from international students was considered but not established as an approved policy in those discussions.

### Unresolved Questions

1. Does International Contract revenue ever transfer to Fund 522?
2. Are there direct contributions from International Programs to student activities instead?
3. Was the proposed international S&A policy subsequently approved?
4. Can annual revenue or transfer amounts be verified from financial records?

### Research Limitations

- Enrollment figures do not establish revenue amounts.
- Fund classifications establish initial accounting treatment, not every subsequent transaction.
- Historical meeting minutes do not establish current policy.
- No confirmed International Contract-to-S&A transfer has yet been identified.

### Official Sources

- https://www.sbctc.edu/colleges-staff/policies-rules/policy-manual/chapter-5.aspx
- https://www.sbctc.edu/colleges-staff/programs-services/accounting-business/clam/student-financials/student-financials-processes.aspx
- https://www.sbctc.edu/colleges-staff/programs-services/accounting-business/rs-ic-revenue-remittance.aspx
- https://www.seattlecolleges.edu/about/policies-and-procedures/pro605?companionId=pro&docID=605&hasboth=1
